# MeterMind — Cross-Model Comparison

Loads mini-eval results from all three models and produces:
- Headline summary table
- Bar chart and radar chart
- Per-metric distributions
- Error analysis (shared failures, DP unchanged outputs, Claude word violations)
- SP vs MA scatter (quality–fluency tradeoff)

No heavy models required — pure pandas + matplotlib.

## 0. Imports

In [ ]:
import json, io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab import files

METRICS  = ['MA', 'SP', 'G']
MODELS   = ['DP Reorderer', 'GRU Attention', 'Claude LLM']
COLOURS  = {'DP Reorderer': 'steelblue', 'GRU Attention': 'seagreen', 'Claude LLM': 'tomato'}
FILES    = {
    'DP Reorderer':  ('mini_dp_reorderer.json',  'mini_dp_reorderer.csv'),
    'GRU Attention': ('mini_gru_attention.json', 'mini_gru_attention.csv'),
    'Claude LLM':    ('mini_claude_llm.json',    'mini_claude_llm.csv'),
}
print('Imports ready.')

## 1. Load results

Upload the 6 files when prompted: 3 JSONs then 3 CSVs.

In [ ]:
print('Upload all 6 files (3 JSON + 3 CSV) when prompted.')
uploaded = files.upload()

summaries = {}   # model -> summary dict
dfs       = {}   # model -> per-line DataFrame

for model, (jf, cf) in FILES.items():
    if jf in uploaded:
        summaries[model] = json.loads(uploaded[jf].decode())
    else:
        print(f'WARNING: {jf} not uploaded — {model} will be skipped.')
    if cf in uploaded:
        dfs[model] = pd.read_csv(io.StringIO(uploaded[cf].decode()))
        dfs[model]['model'] = model
    else:
        print(f'WARNING: {cf} not uploaded — per-line analysis for {model} unavailable.')

# Combined per-line df
all_df = pd.concat(dfs.values(), ignore_index=True) if dfs else pd.DataFrame()

# Summary df
rows = []
for model, s in summaries.items():
    rows.append({'model': model, **{f'{m}_mean': s[m]['mean'] for m in METRICS},
                                  **{f'{m}_std':  s[m]['std']  for m in METRICS}})
summary_df = pd.DataFrame(rows).set_index('model')

print(f'\nLoaded {len(summaries)} summaries and {len(dfs)} per-line datasets.')
print(f'Total lines: {len(all_df)}')

## 2. Headline comparison table

In [ ]:
print(f"{'Model':<18} {'MA mean±std':>14} {'SP mean±std':>14} {'G mean±std':>14}")
print('-' * 64)
for model in MODELS:
    if model not in summaries: continue
    s = summaries[model]
    row = '  '.join(f"{s[m]['mean']:.3f}±{s[m]['std']:.3f}" for m in METRICS)
    print(f"{model:<18} {row}")

print()
for m in METRICS:
    winner = max((model for model in summaries), key=lambda x: summaries[x][m]['mean'])
    print(f"  Best {m}: {winner} ({summaries[winner][m]['mean']:.3f})")

## 3. Bar chart — mean scores with error bars

In [ ]:
present = [m for m in MODELS if m in summaries]
x       = np.arange(len(METRICS))
width   = 0.25

fig, ax = plt.subplots(figsize=(9, 5))
for k, model in enumerate(present):
    s      = summaries[model]
    means  = [s[m]['mean'] for m in METRICS]
    stds   = [s[m]['std']  for m in METRICS]
    offset = (k - len(present)/2 + 0.5) * width
    ax.bar(x + offset, means, width, yerr=stds, capsize=4,
           label=model, color=COLOURS[model], alpha=0.85, ecolor='black', linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(['Metrical Accuracy', 'Semantic Preservation', 'Grammaticality'])
ax.set_ylabel('Mean Score')
ax.set_ylim(0, 1.1)
ax.set_title('MeterMind — Model Comparison (Mini Eval)')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('comparison_bar.png', dpi=150)
plt.show()

## 4. Radar chart — model profiles

In [ ]:
labels  = METRICS
n_axes  = len(labels)
angles  = [n / n_axes * 2 * math.pi * i for i in range(n_axes)]
angles += angles[:1]   # close the polygon

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))
ax.set_theta_offset(math.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(['Metrical\nAccuracy', 'Semantic\nPreservation', 'Grammaticality'], size=11)
ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['0.25', '0.5', '0.75', '1.0'], size=8)
ax.grid(alpha=0.3)

for model in present:
    values  = [summaries[model][m]['mean'] for m in METRICS]
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=model, color=COLOURS[model])
    ax.fill(angles, values, alpha=0.1, color=COLOURS[model])

ax.set_title('Model Profiles', size=13, pad=20)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig('comparison_radar.png', dpi=150)
plt.show()

## 5. Per-metric distributions

In [ ]:
if not all_df.empty:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    fig.suptitle('Score Distributions by Model', fontsize=13)
    for ax, metric in zip(axes, METRICS):
        for model in present:
            if model not in dfs: continue
            vals = dfs[model][metric].dropna()
            ax.hist(vals, bins=15, alpha=0.5, label=model,
                    color=COLOURS[model], edgecolor='white')
            ax.axvline(vals.mean(), color=COLOURS[model],
                       linestyle='--', linewidth=1.5)
        ax.set_title(metric)
        ax.set_xlabel('Score')
        ax.set_ylabel('Count')
        ax.legend(fontsize=8)
    plt.tight_layout()
    plt.savefig('comparison_distributions.png', dpi=150)
    plt.show()

## 6. Error analysis — where each model fails

Bottom 10 lines by MA per model. How much overlap is there — do models fail on the same inputs?

In [ ]:
if not all_df.empty:
    print('=== Bottom 10 lines by MA per model ===')
    worst_inputs = {}
    for model in present:
        if model not in dfs: continue
        worst = dfs[model].nsmallest(10, 'MA')
        worst_inputs[model] = set(worst['input'].tolist())
        print(f'\n{model}:')
        for _, row in worst.iterrows():
            print(f"  MA={row['MA']:.3f}  {row['input'][:60]}")

    # Overlap
    print('\n=== Failure overlap (shared worst inputs) ===')
    models_with_data = [m for m in present if m in worst_inputs]
    for i, m1 in enumerate(models_with_data):
        for m2 in models_with_data[i+1:]:
            shared = worst_inputs[m1] & worst_inputs[m2]
            print(f"  {m1} ∩ {m2}: {len(shared)} shared failures")
            for s in shared:
                print(f"    {s[:70]}")

## 7. DP unchanged output analysis

How often does the DP reorderer return the input unchanged? Do those lines score differently?

In [ ]:
if 'DP Reorderer' in dfs:
    dp = dfs['DP Reorderer'].copy()
    # Normalise both sides for comparison
    dp['input_clean']  = dp['input'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()
    dp['output_clean'] = dp['output'].str.lower().str.replace(r'[^a-z\s]', '', regex=True).str.strip()
    dp['unchanged']    = dp['input_clean'] == dp['output_clean']

    n_unchanged = dp['unchanged'].sum()
    print(f'DP unchanged outputs: {n_unchanged} / {len(dp)} ({100*n_unchanged/len(dp):.1f}%)')
    print()

    for group, label in [(True, 'Unchanged'), (False, 'Reordered')]:
        subset = dp[dp['unchanged'] == group]
        if len(subset) == 0: continue
        print(f'{label} (n={len(subset)}):')
        for m in METRICS:
            print(f"  {m}: {subset[m].mean():.3f} ± {subset[m].std():.3f}")
        print()

## 8. Claude word constraint analysis

For each Claude output, count words added that weren't in the input. Does violating the constraint correlate with better or worse MA?

In [ ]:
if 'Claude LLM' in dfs:
    cl = dfs['Claude LLM'].copy()

    def violation_count(row):
        inp  = set(row['input'].lower().split())
        out  = set(row['output'].lower().split())
        return len(out - inp)

    cl['violations'] = cl.apply(violation_count, axis=1)
    n_viol = (cl['violations'] > 0).sum()
    print(f'Claude outputs with word violations: {n_viol} / {len(cl)} ({100*n_viol/len(cl):.1f}%)')
    print(f'Mean violations per output: {cl["violations"].mean():.2f}')
    print()

    # Compliant vs violating
    for group, label in [(0, 'Compliant (no new words)'), (None, 'Violations (added words)')]:
        subset = cl[cl['violations'] == 0] if group == 0 else cl[cl['violations'] > 0]
        if len(subset) == 0: continue
        print(f'{label} (n={len(subset)}):')
        for m in METRICS:
            print(f"  {m}: {subset[m].mean():.3f} ± {subset[m].std():.3f}")
        print()

    # Scatter: violations vs MA
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.scatter(cl['violations'], cl['MA'], alpha=0.6, color=COLOURS['Claude LLM'], edgecolors='white')
    ax.set_xlabel('Number of added words (violations)')
    ax.set_ylabel('Metrical Accuracy')
    ax.set_title('Claude: Word Constraint Violations vs MA')
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('claude_violations.png', dpi=150)
    plt.show()

## 9. SP vs MA scatter — quality–fluency tradeoff

Does improving metrical accuracy come at the cost of semantic preservation?

In [ ]:
if not all_df.empty:
    fig, ax = plt.subplots(figsize=(7, 5))
    for model in present:
        if model not in dfs: continue
        d = dfs[model]
        ax.scatter(d['MA'], d['SP'], alpha=0.5, label=model,
                   color=COLOURS[model], edgecolors='white', s=50)
    ax.set_xlabel('Metrical Accuracy (MA)')
    ax.set_ylabel('Semantic Preservation (SP)')
    ax.set_title('Quality–Fluency Tradeoff: MA vs SP')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig('comparison_scatter.png', dpi=150)
    plt.show()

    # Correlation per model
    print('MA–SP correlation per model:')
    for model in present:
        if model not in dfs: continue
        r = dfs[model][['MA', 'SP']].corr().iloc[0, 1]
        print(f'  {model}: r={r:.3f}')

## 10. Save comparison summary

In [ ]:
# Combined summary JSON
comparison_summary = {'models': summaries}
with open('comparison_summary.json', 'w') as f:
    json.dump(comparison_summary, f, indent=2)
print('Saved comparison_summary.json')

# Combined per-line CSV
if not all_df.empty:
    all_df.to_csv('comparison_all_lines.csv', index=False)
    print('Saved comparison_all_lines.csv')

# Download everything
for fname in ['comparison_summary.json', 'comparison_all_lines.csv',
              'comparison_bar.png', 'comparison_radar.png',
              'comparison_distributions.png', 'comparison_scatter.png']:
    try:
        files.download(fname)
    except Exception:
        pass
print('Done.')